# Neural ODEs Exploration

This notebook demonstrates the capabilities of Neural Ordinary Differential Equations (Neural ODEs) for continuous-time modeling.

## Key Features:
- **Memory Efficiency**: Constant memory usage regardless of network depth
- **Continuous-Time Modeling**: Natural handling of irregular time sampling  
- **Flexible Architectures**: Can model complex dynamical systems
- **Interpretability**: ODE structure provides physical insights

## Safety Notice
⚠️ This is for research/educational purposes only. Not for production use.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torchdiffeq import odeint

# Import our models
import sys
sys.path.append('../src')
from models import BasicODEFunc, NeuralODE, LinearBaseline, MLPBaseline

print("Imports successful!")


In [ ]:
def create_simple_dataset():
    """Create a simple sine wave dataset."""
    t = torch.linspace(0, 4*np.pi, 100)
    y = torch.sin(t) + 0.1 * torch.randn_like(t)
    return t, y

# Create dataset
print("Creating dataset...")
times, values = create_simple_dataset()

# Plot the dataset
plt.figure(figsize=(10, 4))
plt.plot(times.numpy(), values.numpy(), 'b-', alpha=0.8)
plt.xlabel('Time')
plt.ylabel('Value')
plt.title('Synthetic Sine Wave Dataset')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Dataset created with {len(times)} points")
print(f"Time range: [{times.min():.2f}, {times.max():.2f}]")
print(f"Value range: [{values.min():.2f}, {values.max():.2f}]")


In [ ]:
def train_model(model, times, values, epochs=1000):
    """Train a model on the dataset."""
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()
    
    losses = []
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        
        if hasattr(model, 'neural_ode'):
            # Neural ODE
            x0 = values[0:1].unsqueeze(0)
            predictions = model(x0, times)
            loss = criterion(predictions.squeeze(), values)
        else:
            # Regular neural network
            predictions = model(times.unsqueeze(-1))
            loss = criterion(predictions.squeeze(), values)
        
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        
        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.6f}")
    
    return losses

# Create models
print("Creating models...")
models = {
    'Linear': LinearBaseline(input_dim=1, output_dim=1),
    'MLP': MLPBaseline(input_dim=1, hidden_dims=[32, 16], output_dim=1),
    'Neural ODE': NeuralODE(
        ode_func=BasicODEFunc(hidden_dim=32, activation='tanh'),
        rtol=1e-3, atol=1e-4, method='dopri5'
    )
}

print(f"Created {len(models)} models for comparison")


In [ ]:
# Train and evaluate models
results = {}
for name, model in models.items():
    print(f"\nTraining {name}...")
    losses = train_model(model, times, values, epochs=500)
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        if hasattr(model, 'neural_ode'):
            x0 = values[0:1].unsqueeze(0)
            predictions = model(x0, times)
        else:
            predictions = model(times.unsqueeze(-1))
    
    results[name] = {
        'model': model,
        'predictions': predictions.squeeze(),
        'losses': losses
    }
    
    print(f"{name} training completed!")

print("\nAll models trained successfully!")


In [ ]:
# Compare all models
print("Comparing all models...")
plt.figure(figsize=(15, 10))

# Plot predictions
plt.subplot(2, 2, 1)
plt.plot(times.numpy(), values.numpy(), 'b-', label='Ground Truth', alpha=0.8, linewidth=2)
colors = ['red', 'green', 'orange']
for i, (name, result) in enumerate(results.items()):
    plt.plot(times.numpy(), result['predictions'].detach().numpy(), 
             '--', color=colors[i], label=f'{name}', alpha=0.8, linewidth=2)
plt.xlabel('Time')
plt.ylabel('Value')
plt.title('All Model Predictions')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot training losses
plt.subplot(2, 2, 2)
for i, (name, result) in enumerate(results.items()):
    plt.plot(result['losses'], color=colors[i], label=f'{name}', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Losses')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')

# Plot residuals
plt.subplot(2, 2, 3)
for i, (name, result) in enumerate(results.items()):
    residuals = result['predictions'].detach().numpy() - values.numpy()
    plt.plot(times.numpy(), residuals, color=colors[i], label=f'{name}', alpha=0.8, linewidth=2)
plt.xlabel('Time')
plt.ylabel('Residuals')
plt.title('Prediction Residuals')
plt.legend()
plt.grid(True, alpha=0.3)

# Compute and display metrics
plt.subplot(2, 2, 4)
metrics_data = []
for name, result in results.items():
    pred = result['predictions'].detach().numpy()
    target = values.numpy()
    
    mse = np.mean((pred - target) ** 2)
    mae = np.mean(np.abs(pred - target))
    rmse = np.sqrt(mse)
    
    metrics_data.append([name, f"{mse:.6f}", f"{mae:.6f}", f"{rmse:.6f}"])

# Create metrics table
metrics_table = plt.table(
    cellText=metrics_data,
    colLabels=['Model', 'MSE', 'MAE', 'RMSE'],
    cellLoc='center',
    loc='center'
)
metrics_table.auto_set_font_size(False)
metrics_table.set_fontsize(10)
metrics_table.scale(1.2, 1.5)
plt.axis('off')
plt.title('Performance Metrics')

plt.tight_layout()
plt.show()

print("\nModel Comparison Complete!")
print("Neural ODEs provide a powerful way to model continuous-time dynamics.")
print("They can capture complex temporal patterns while being memory efficient.")
